# Install Dependencies and Import

In [2]:
# Install dependencies
!pip install -q --upgrade numerapi numerai-tools optuna optuna-integration[lightgbm]

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from datetime import timedelta
import time

from numerapi import NumerAPI
from numerai_tools.scoring import numerai_corr, correlation_contribution, neutralize

import json
import os
import gc
import shutil
import itertools
from tqdm import tqdm
import random

import lightgbm as lgb
from lightgbm.callback import early_stopping, log_evaluation
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV, cross_validate
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import make_scorer
import cloudpickle
import pickle
import optuna
from optuna import Trial
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner
from optuna.visualization.matplotlib import plot_optimization_history, plot_param_importances, plot_contour


import warnings
warnings.filterwarnings('ignore')

pd.options.display.max_columns = 500

# Inline plots
%matplotlib inline

SEED = 42
random.seed(SEED); np.random.seed(SEED)

# Functions

In [4]:
def per_era_metrics(val_df, pred_col, calc_mmc=True, only_metrics=True):
    """Расчёт метрик по эрам: CORR и MMC"""

    # Вспомогательные функции
    def sharpe(series): 
        std = series.std(ddof=0)
        return series.mean() / std if std != 0 else np.nan

    def max_drawdown(series):
        cum = series.cumsum()
        return (cum.expanding().max() - cum).max()

    # Словарь для хранения метрик
    metrics = {}

    # CORR
    per_era_corr = val_df.groupby("era").apply(lambda x: numerai_corr(x[[pred_col]], x["target"]))

    metrics.update({
        "corr_mean": round(per_era_corr[pred_col].mean(), 3),
        "corr_std": round(per_era_corr[pred_col].std(ddof=0), 3),
        "corr_sharpe": round(sharpe(per_era_corr[pred_col]), 3),
        "corr_max_drawdown": round(max_drawdown(per_era_corr[pred_col]), 3),
    })

    # MMC
    if calc_mmc:
        per_era_mmc = val_df.dropna().groupby("era").apply(lambda x: correlation_contribution(x[[pred_col]], x["meta_model"], x["target"]))

        metrics.update({
            "mmc_mean": round(per_era_mmc[pred_col].mean(), 3),
            "mmc_std": round(per_era_mmc[pred_col].std(ddof=0), 3),
            "mmc_sharpe": round(sharpe(per_era_mmc[pred_col]), 3),
            "mmc_max_drawdown": round(max_drawdown(per_era_mmc[pred_col]), 3),
        })

    if only_metrics:
        return metrics
    if calc_mmc:
        return metrics, per_era_corr, per_era_mmc
    else:
        return metrics, per_era_corr

In [5]:
def train_and_evaluate(train_df, val_df, feature_set, model_name, params) -> tuple:

    start_time = time.time()

    # Обучение модели
    model = lgb.LGBMRegressor(**params, random_state=SEED)
    model.fit(train_df[feature_set], train_df["target"])
    print("Обучение завершено")

    # Прогноз на Validation
    val_df[model_name] = model.predict(val_df[feature_set])
    print("Валидация завершена")

    # Расчёт времени
    elapsed_time = str(timedelta(seconds=int(time.time() - start_time)))

    return elapsed_time, model

In [6]:
def era_wise_cv(train_df, feature_set, params, embargo=4, n_splits=5):
    eras = sorted(train_df["era"].unique())
    tscv = TimeSeriesSplit(n_splits=n_splits, gap=embargo)

    results = []
    all_importances = []

    for fold, (train_idx, val_idx) in enumerate(tscv.split(eras), 1):
        train_eras = [eras[i] for i in train_idx]
        val_eras = [eras[i] for i in val_idx]

        # print(f"\nFold {fold}: Train {train_eras[0]}–{train_eras[-1]}, Val {val_eras[0]}–{val_eras[-1]}")

        train_split = train_df[train_df["era"].isin(train_eras)].copy()
        val_split = train_df[train_df["era"].isin(val_eras)].copy()
        
        X_train, y_train = train_split[feature_set], train_split["target"]
        X_val, y_val = val_split[feature_set], val_split["target"] 


        model = lgb.LGBMRegressor(**params, random_state=SEED)
        model.fit(X_train, y_train)
        val_split["prediction"] = model.predict(X_val)

        per_era_corr = val_split.groupby("era").apply(lambda x: numerai_corr(x[["prediction"]], x["target"]))["prediction"] 
        corr_mean = per_era_corr.mean()
        corr_std = per_era_corr.std(ddof=0)
        corr_sharpe = corr_mean / corr_std if corr_std != 0 else np.nan

        results.append({
            "fold": str(fold),
            "train_eras": f"{train_eras[0]}-{train_eras[-1]}",
            "val_eras": f"{val_eras[0]}-{val_eras[-1]}",
            "corr_sharpe": round(corr_sharpe, 3),
            "corr_mean": round(corr_mean, 3),
            "corr_std": round(corr_std, 3),
        })
        

        importance = pd.DataFrame({
            "feature": feature_set,
            "importance": model.feature_importances_})
        all_importances.append(importance.set_index("feature"))


        del model
        gc.collect()


    # Сборка результатов
    cv_results = pd.DataFrame(results)
    cv_mean = cv_results.mean(numeric_only=True).round(3).to_dict()
    
    # Сборка важности признаков
    importance_df = pd.concat(all_importances, axis=1).mean(axis=1).sort_values(ascending=False).reset_index()
    importance_df.columns = ["feature", "importance"]


    return cv_results, cv_mean, importance_df

In [7]:
def era_wise_cv_optuna(train_df, feature_set, params, embargo=4, n_splits=3, esr=None):

    eras = sorted(train_df["era"].unique())
    tscv = TimeSeriesSplit(n_splits=n_splits, gap=embargo)
    
    results = []       
    best_iters = []    

    for fold, (train_idx, val_idx) in enumerate(tscv.split(eras), 1):
        
        train_eras = [eras[i] for i in train_idx]
        val_eras = [eras[i] for i in val_idx]

        train_split = train_df[train_df["era"].isin(train_eras)]
        val_split = train_df[train_df["era"].isin(val_eras)]

        X_train, y_train = train_split[feature_set], train_split["target"]
        X_val, y_val = val_split[feature_set], val_split["target"]

        model = lgb.LGBMRegressor(**params, random_state=SEED)

        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            eval_names=["val"],
            callbacks=[
                lgb.early_stopping(stopping_rounds=esr, verbose=False),
                lgb.log_evaluation(period=0) 
            ],
        )

        best_iters.append(model.best_iteration_)

        val_split = val_split.copy()
        val_split["prediction"] = model.predict(X_val)

    
        per_era_corr = val_split.groupby("era").apply(lambda x: numerai_corr(x[["prediction"]], x["target"]))["prediction"]
        corr_mean = per_era_corr.mean()
        corr_std = per_era_corr.std(ddof=0)
        corr_sharpe = corr_mean / corr_std if corr_std > 0 else np.nan
        
        results.append(corr_sharpe)

    return {"corr_sharpe": np.mean(results), "best_iteration": int(np.mean(best_iters))}


In [8]:
def get_feature_importance(model, normalize=False):
    
    feat_names = model.booster_.feature_name()
    gain  = model.booster_.feature_importance(importance_type="gain")
    split = model.booster_.feature_importance(importance_type="split")

    df = pd.DataFrame({"feature": feat_names, "gain": gain, "split": split})

    if normalize:
        df["gain"]  = df["gain"]  / df["gain"].sum()
        df["split"] = df["split"] / df["split"].sum()

    return df.sort_values("gain", ascending=False).reset_index(drop=True)

In [9]:
def get_diff_cv(results_df, model_name_diff, model_name_cv, cv_mean):
    row = results_df.loc[results_df["model_name"] == model_name_diff, ["model_name", "corr_sharpe", "corr_mean", "corr_std"]]
    row_cv = pd.DataFrame([{"model_name": model_name_cv, "corr_sharpe": cv_mean["corr_sharpe"], "corr_mean": cv_mean["corr_mean"], "corr_std": cv_mean["corr_std"],}])
    df_diff_cv = pd.concat([row, row_cv], ignore_index=True)

    return df_diff_cv

In [10]:
def log_to_results(results_df, model_name, elapsed_time, metrics, params):
    new_row = {"model_name": model_name, "time": elapsed_time, **metrics, **params}
    results_df = pd.concat([results_df, pd.DataFrame([new_row])], ignore_index=True)

    return results_df

In [11]:
def get_last_fold(train_df, feature_set, fold=-1):
    eras = sorted(train_df["era"].unique())
    tscv = TimeSeriesSplit(n_splits=5, gap=4)
    splits = list(tscv.split(eras))
    train_idx, val_idx = splits[fold]

    last_train_eras = [eras[i] for i in train_idx]
    last_val_eras = [eras[i] for i in val_idx]

    train_es = train_df[train_df["era"].isin(last_train_eras)]
    val_es = train_df[train_df["era"].isin(last_val_eras)]

    X_train_es = train_es[feature_set]
    y_train_es = train_es["target"]
    X_val_es = val_es[feature_set]
    y_val_es = val_es["target"]

    return X_train_es, y_train_es, X_val_es, y_val_es

In [12]:
def save_experiment(model, model_name, validation_data, results_df):
    # Имена файлов
    model_file = f"/content/drive/MyDrive/Colab Notebooks/Data/{model_name}.pkl"
    validation_file = "/content/drive/MyDrive/Colab Notebooks/Data/val_med_pred.pkl"
    results_file = "/content/drive/MyDrive/Colab Notebooks/Data/results_med.pkl"

    # Сохранение
    with open(model_file, "wb") as f:
        pickle.dump(model, f)
 
    with open(validation_file, "wb") as f:
        pickle.dump(validation_data, f)

    with open(results_file, "wb") as f:
        pickle.dump(results_df, f)

# Loading Numerai Datasets  

В предыдущем блокноте:
- Почистил `train` от эр с пропусками.
- Применил `validation` эмбарго 4 эры и добавил `numerai_meta_model`.
- Сгруппировал признаки по наборам для `medium`.  
  
Загружаю с Google Disk.

In [13]:
# Загрузка подготовленных датасетов
train = pd.read_parquet("/content/drive/MyDrive/Colab Notebooks/train_medium.parquet")
validation = pd.read_parquet("/content/drive/MyDrive/Colab Notebooks/validation_medium.parquet")

# Загрузка метаданных
with open('/content/drive/MyDrive/Colab Notebooks/dict_medium_clean.pkl', 'rb') as f:
    dict_medium = pickle.load(f)

# Список признаков
feature_set = list(dict_medium['all'])


print(f"Train: {train.shape}, Eras: {train['era'].min()}-{train['era'].max()}")
print(f"Validation: {validation.shape}, Eras: {validation['era'].min()}-{validation['era'].max()}")
print(f"Features: {len(feature_set)}")

Train: (1865208, 739), Eras: 210-574
Validation: (3798341, 740), Eras: 579-1190
Features: 737


In [14]:
# Датафреймы для результатов
cols = ["model_name", "time"]
cols_corr = ["corr_sharpe", "corr_mean", "corr_std", "corr_max_drawdown"]
cols_mmc = ["mmc_mean", "mmc_std", "mmc_sharpe", "mmc_max_drawdown"]


results_df = pd.DataFrame(columns=(cols + cols_corr + cols_mmc))
neut_df = results_df.copy()  # Датафрейм для нейтрализованных результатов
results_df

,model_name,time,corr_sharpe,corr_mean,corr_std,corr_max_drawdown,mmc_mean,mmc_std,mmc_sharpe,mmc_max_drawdown


In [ ]:
# with open("/content/drive/MyDrive/Colab Notebooks/Data/val_med_pred.pkl", "rb") as f:  # Загрузка validation
#     validation = pickle.load(f)

# with open("/content/drive/MyDrive/Colab Notebooks/Data/results_med.pkl", "rb") as f:  # Загрузка results_df
#     results_df = pickle.load(f)

In [ ]:
results_df

,model_name,time,corr_sharpe,corr_mean,corr_std,corr_max_drawdown,mmc_mean,mmc_std,mmc_sharpe,mmc_max_drawdown,n_estimators,learning_rate,max_depth,num_leaves,colsample_bytree,verbosity,device_type,reg_alpha,reg_lambda,min_gain_to_split,subsample,min_child_samples,min_child_weight
0,baseline,1:20:42,1.472,0.031,0.021,0.060,-0.002,0.011,-0.173,0.201,20000.0,0.001,6.0,64.0,0.10,-1.0,gpu,NaN,NaN,NaN,NaN,NaN,NaN
1,tuned_0,0:29:41,1.351,0.030,0.022,0.078,-0.003,0.011,-0.230,0.240,10069.0,0.001,6.0,64.0,0.10,-1.0,gpu,1.0,1.0,0.01,0.9,NaN,NaN
2,tuned,0:19:53,1.530,0.030,0.020,0.066,-0.001,0.010,-0.088,0.124,10000.0,0.010,5.0,31.0,0.05,-1.0,gpu,10.0,10.0,NaN,0.8,200.0,0.01


# Baseline Model with Large Params Numerai  

In [15]:
# LGBM Regressor 
model_name = "baseline"

params = {
    "n_estimators": 20000,
    "learning_rate": 0.001,
    "max_depth": 6,
    "num_leaves": 64,
    "colsample_bytree": 0.1,
    "verbosity": -1,
    "device_type": "gpu",
}

elapsed_time, model = train_and_evaluate(train, validation, feature_set, model_name=model_name, params=params)
metrics = per_era_metrics(validation, pred_col=model_name)
results_df = log_to_results(results_df, model_name=model_name, elapsed_time=elapsed_time, metrics=metrics, params=params)
save_experiment(model=model, model_name=model_name, validation_data=validation, results_df=results_df)
results_df

Обучение завершено
Валидация завершена


,model_name,time,corr_sharpe,corr_mean,corr_std,corr_max_drawdown,mmc_mean,mmc_std,mmc_sharpe,mmc_max_drawdown,n_estimators,learning_rate,max_depth,num_leaves,colsample_bytree,verbosity,device_type
0,baseline,1:20:42,1.472,0.031,0.021,0.06,-0.002,0.011,-0.173,0.201,20000.0,0.001,6.0,64.0,0.1,-1.0,gpu


# Era-Wise CV for Baseline

In [17]:
model_name = "baseline_cv"

cv_results, cv_mean, _ = era_wise_cv(train, feature_set, params, n_splits=5)

print("РЕЗУЛЬТАТЫ ПО ФОЛДАМ:")
display(cv_results)
print("СРАВНЕНИЕ С БАЗОВОЙ МОДЕЛЬЮ:")
display(get_diff_cv(results_df, model_name_diff="baseline", model_name_cv="baseline_cv", cv_mean=cv_mean))

РЕЗУЛЬТАТЫ ПО ФОЛДАМ:


,fold,train_eras,val_eras,corr_sharpe,corr_mean,corr_std
0,1,210-270,275-334,1.652,0.031,0.019
1,2,210-330,335-394,1.926,0.029,0.015
2,3,210-390,395-454,2.607,0.043,0.016
3,4,210-450,455-514,1.859,0.035,0.019
4,5,210-510,515-574,2.527,0.042,0.017


СРАВНЕНИЕ С БАЗОВОЙ МОДЕЛЬЮ:


,model_name,corr_sharpe,corr_mean,corr_std
0,baseline,1.472,0.031,0.021
1,baseline_cv,2.114,0.036,0.017


# Early Stopping on Last Fold

In [24]:
# Получаем best_iter для последнего фолда
esr = 100

X_train_es, y_train_es, X_val_es, y_val_es = get_last_fold(train, feature_set, fold=-1)

model = lgb.LGBMRegressor(**params, random_state=SEED)
model.fit(
    X_train_es, y_train_es,
    eval_set=[(X_val_es, y_val_es)],
    eval_names=["val_es"],
    callbacks=[early_stopping(stopping_rounds=esr), log_evaluation(period=0)]
)

best_iter_last_fold = model.best_iteration_
print(f"Best Iter Last Fold: {best_iter_last_fold}")

Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[8391]	val_es's l2: 0.0496948
Best Iter Last Fold: 8391


# Tuned Params

In [ ]:
# Holdout Validation with Best Iteration + Регуляризация
model_name = "tuned_0"

params_tuned_0 = {
    "n_estimators": int(best_iter_last_fold * 1.2), 
    "learning_rate": 0.001,
    "max_depth": 6,
    "num_leaves": 64,
    "colsample_bytree": 0.1,
    "reg_alpha": 1.0,
    "reg_lambda": 1.0,
    "min_gain_to_split": 0.01,
    "subsample": 0.9,
    "verbosity": -1,
    "device_type": "gpu",
}

elapsed_time, model = train_and_evaluate(train, validation, feature_set, model_name=model_name, params=params_tuned_0)
metrics = per_era_metrics(validation, pred_col=model_name)
results_df = log_to_results(results_df, model_name=model_name, elapsed_time=elapsed_time, metrics=metrics, params=params_tuned_0)
save_experiment(model=model, model_name=model_name, validation_data=validation, results_df=results_df)
results_df

,model_name,time,corr_sharpe,corr_mean,corr_std,corr_max_drawdown,mmc_mean,mmc_std,mmc_sharpe,mmc_max_drawdown,n_estimators,learning_rate,max_depth,num_leaves,colsample_bytree,verbosity,device_type,reg_alpha,reg_lambda,min_gain_to_split,subsample,min_child_samples,min_child_weight
0,baseline,1:20:42,1.472,0.031,0.021,0.060,-0.002,0.011,-0.173,0.201,20000.0,0.001,6.0,64.0,0.1,-1.0,gpu,NaN,NaN,NaN,NaN,NaN,NaN
1,tuned_0,0:29:41,1.351,0.030,0.022,0.078,-0.003,0.011,-0.230,0.240,10069.0,0.001,6.0,64.0,0.1,-1.0,gpu,1.0,1.0,0.01,0.9,NaN,NaN


In [ ]:
# Holdout Validation with Best Iteration + Регуляризация
model_name = "tuned"

params_tuned = {
    'n_estimators': 10000,    
    'learning_rate': 0.01,           
    'max_depth': 5,                  
    'num_leaves': 31,                
    'colsample_bytree': 0.05,        
    'subsample': 0.8,               
    'reg_alpha': 10.0,               
    'reg_lambda': 10.0,             
    'min_child_samples': 200,        
    'min_child_weight': 1e-2,       
    'verbosity': -1,
    'device_type': 'gpu',
}

elapsed_time, model = train_and_evaluate(train, validation, feature_set, model_name=model_name, params=params_tuned)
metrics = per_era_metrics(validation, pred_col=model_name)
results_df = log_to_results(results_df, model_name=model_name, elapsed_time=elapsed_time, metrics=metrics, params=params_tuned)
save_experiment(model=model, model_name=model_name, validation_data=validation, results_df=results_df)
results_df

Обучение завершено
Валидация завершена


,model_name,time,corr_sharpe,corr_mean,corr_std,corr_max_drawdown,mmc_mean,mmc_std,mmc_sharpe,mmc_max_drawdown,...,num_leaves,colsample_bytree,verbosity,device_type,reg_alpha,reg_lambda,min_gain_to_split,subsample,min_child_samples,min_child_weight
0,baseline,1:20:42,1.472,0.031,0.021,0.060,-0.002,0.011,-0.173,0.201,...,64.0,0.10,-1.0,gpu,NaN,NaN,NaN,NaN,NaN,NaN
1,tuned_0,0:29:41,1.351,0.030,0.022,0.078,-0.003,0.011,-0.230,0.240,...,64.0,0.10,-1.0,gpu,1.0,1.0,0.01,0.9,NaN,NaN
2,tuned,0:19:53,1.530,0.030,0.020,0.066,-0.001,0.010,-0.088,0.124,...,31.0,0.05,-1.0,gpu,10.0,10.0,NaN,0.8,200.0,0.01


# Optuna

In [ ]:
def objective(trial):

    params_optuna = {
        # --- Основные параметры ---
        'n_estimators': trial.suggest_int('n_estimators', 8000, 12000, step=500),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.02, log=True),
        'max_depth': trial.suggest_int('max_depth', 4, 7),
        'num_leaves': trial.suggest_int('num_leaves', 15, 63, step=8),
        
        # --- Регуляризация ---
        'reg_alpha': trial.suggest_float('reg_alpha', 1.0, 20.0, log=True),  
        'reg_lambda': trial.suggest_float('reg_lambda', 1.0, 20.0, log=True),  
        'min_gain_to_split': trial.suggest_float('min_gain_to_split', 0.0, 0.1),
        'min_child_samples': trial.suggest_int('min_child_samples', 100, 400, step=50),
        
        # --- Сэмплирование ---
        'subsample': trial.suggest_float('subsample', 0.7, 0.9),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.03, 0.12),

        'verbosity': -1,
        'device_type': 'gpu',
    }

    # Запускаем кросс-валидацию
    cv_result = era_wise_cv_optuna(
        train_df=train,
        feature_set=feature_set,
        params=params_optuna,
        n_splits=3,
        esr=100 
    )

    trial.set_user_attr("best_iteration", cv_result["best_iteration"])

    return cv_result["corr_sharpe"]

In [ ]:
# --- ЗАПУСК ПОДБОРА ГИПЕРПАРАМЕТРОВ --- 
# 210 min

optuna.logging.set_verbosity(optuna.logging.WARNING)

study = optuna.create_study(direction="maximize", study_name="lgbm-tuning")

# Запускаем оптимизацию
study.optimize(objective, n_trials=50)

results = study.trials_dataframe()
results = results.sort_values("value", ascending=False)

print("Best Params:", study.best_params)
print("Best Corr Sharpe:", study.best_value)
print("Best Iteration:", study.best_trial.user_attrs["best_iteration"])

display(results[["number", "value", "state"] + [c for c in results.columns if "params_" in c]].head(10))

Best Params: {'n_estimators': 11500, 'learning_rate': 0.005840171633621238, 'max_depth': 7, 'num_leaves': 63, 'reg_alpha': 15.331299559116657, 'reg_lambda': 19.854539593326436, 'min_gain_to_split': 0.07415059548192905, 'min_child_samples': 100, 'subsample': 0.7765851706305711, 'colsample_bytree': 0.10815581236829545}
Best Corr Sharpe: 1.9154744891448436
Best Iteration: 1091


,number,value,state,params_colsample_bytree,params_learning_rate,params_max_depth,params_min_child_samples,params_min_gain_to_split,params_n_estimators,params_num_leaves,params_reg_alpha,params_reg_lambda,params_subsample
46,46,1.915474,COMPLETE,0.108156,0.005840,7,100,0.074151,11500,63,15.331300,19.854540,0.776585
45,45,1.904810,COMPLETE,0.096557,0.006677,7,100,0.073741,11500,55,8.016404,12.108561,0.741470
44,44,1.888838,COMPLETE,0.094920,0.009494,7,100,0.064222,10500,63,7.949611,13.241606,0.739896
38,38,1.888771,COMPLETE,0.082535,0.012540,6,100,0.049907,10500,63,4.046404,6.241561,0.716947
15,15,1.885646,COMPLETE,0.081486,0.008012,5,300,0.039999,9000,55,16.768980,6.620855,0.720246
10,10,1.883666,COMPLETE,0.034455,0.009215,5,300,0.049045,9500,63,7.150553,2.540617,0.707457
43,43,1.882697,COMPLETE,0.095605,0.008609,6,150,0.062339,10500,55,8.067307,12.007002,0.738339
35,35,1.872006,COMPLETE,0.072553,0.009286,5,100,0.045769,10500,63,6.220245,2.963087,0.724157
12,12,1.871729,COMPLETE,0.031264,0.008519,5,300,0.058651,9500,55,6.152344,3.646876,0.703593
11,11,1.869671,COMPLETE,0.033886,0.009089,5,300,0.047455,9500,63,7.010253,2.381608,0.700048


In [ ]:
params_best_optuna3 = {
    'n_estimators': 9000, 
    'learning_rate': 0.008, 
    'max_depth': 5, 
    'num_leaves': 55, 
    'reg_alpha': 16, 
    'reg_lambda': 6, 
    'min_gain_to_split': 0.03, 
    'min_child_samples': 300, 
    'subsample': 0.72, 
    'colsample_bytree': 0.08,
    'verbosity': -1,
    'device_type': "gpu",
    }

In [ ]:
# print(f"Tuned Model Corr Sharpe: {results_df.loc[results_df['model_name'] == 'tuned', 'corr_sharpe'].values[0]}")
# print(f"Optuna Best Corr Sharpe: {study.best_trial.value}")

# plot_optimization_history(study)
# plot_param_importances(study)
# plot_contour(study, params=['learning_rate', 'max_depth']) 
# plt.show()

# Model with Best Params Optuna

In [24]:
# LGBM Regressor 
model_name = "best_optuna"

# best_params = study.best_params.copy()
# best_params.update({
#     'n_estimators': study.best_trial.user_attrs.get('best_iteration', 500),
#     'verbose': -1
# })

params_best_optuna = {
    'n_estimators': 11500, 
    'learning_rate': 0.005, 
    'max_depth': 7, 
    'num_leaves': 63, 
    'reg_alpha': 15, 
    'reg_lambda': 19, 
    'min_gain_to_split': 0.07, 
    'min_child_samples': 100, 
    'subsample': 0.7, 
    'colsample_bytree': 0.1,
    'verbosity': -1,
    'device_type': "gpu",
    }


elapsed_time, model = train_and_evaluate(train, validation, feature_set, model_name=model_name, params=params_best_optuna)
metrics = per_era_metrics(validation, pred_col=model_name)
results_df = log_to_results(results_df, model_name=model_name, elapsed_time=elapsed_time, metrics=metrics, params=params_best_optuna)
save_experiment(model=model, model_name=model_name, validation_data=validation, results_df=results_df)
results_df

Обучение завершено
Валидация завершена


,model_name,time,corr_sharpe,corr_mean,corr_std,corr_max_drawdown,mmc_mean,mmc_std,mmc_sharpe,mmc_max_drawdown,n_estimators,learning_rate,max_depth,num_leaves,colsample_bytree,verbosity,device_type,reg_alpha,reg_lambda,min_gain_to_split,subsample,min_child_samples,min_child_weight
0,baseline,1:20:42,1.472,0.031,0.021,0.060,-0.002,0.011,-0.173,0.201,20000.0,0.001,6.0,64.0,0.10,-1.0,gpu,NaN,NaN,NaN,NaN,NaN,NaN
1,tuned_0,0:29:41,1.351,0.030,0.022,0.078,-0.003,0.011,-0.230,0.240,10069.0,0.001,6.0,64.0,0.10,-1.0,gpu,1.0,1.0,0.01,0.9,NaN,NaN
2,tuned,0:19:53,1.530,0.030,0.020,0.066,-0.001,0.010,-0.088,0.124,10000.0,0.010,5.0,31.0,0.05,-1.0,gpu,10.0,10.0,NaN,0.8,200.0,0.01
3,best_optuna,0:24:27,1.493,0.031,0.021,0.064,-0.001,0.010,-0.109,0.141,11500.0,0.005,7.0,63.0,0.10,-1.0,gpu,15.0,19.0,0.07,0.7,100.0,NaN


# Model with Best Params Optuna2

In [ ]:
# LGBM Regressor 
model_name = "best_optuna2"

params_best_optuna2 = {
    'n_estimators': 11500, 
    'learning_rate': 0.006, 
    'max_depth': 7, 
    'num_leaves': 55, 
    'reg_alpha': 8, 
    'reg_lambda': 12, 
    'min_gain_to_split': 0.07, 
    'min_child_samples': 100, 
    'subsample': 0.7, 
    'colsample_bytree': 0.09,
    'verbosity': -1,
    'device_type': "gpu",
    }


elapsed_time, model = train_and_evaluate(train, validation, feature_set, model_name=model_name, params=params_best_optuna2)
metrics = per_era_metrics(validation, pred_col=model_name)
results_df = log_to_results(results_df, model_name=model_name, elapsed_time=elapsed_time, metrics=metrics, params=params_best_optuna2)
save_experiment(model=model, model_name=model_name, validation_data=validation, results_df=results_df)
results_df

Обучение завершено


# Model with Best Params Optuna3

In [ ]:
# LGBM Regressor 
model_name = "best_optuna3"

params_best_optuna3 = {
    'n_estimators': 9000, 
    'learning_rate': 0.008, 
    'max_depth': 5, 
    'num_leaves': 31, 
    'reg_alpha': 16, 
    'reg_lambda': 6, 
    'min_gain_to_split': 0.03, 
    'min_child_samples': 300, 
    'subsample': 0.72, 
    'colsample_bytree': 0.08,
    'verbosity': -1,
    'device_type': "gpu",
    }


elapsed_time, model = train_and_evaluate(train, validation, feature_set, model_name=model_name, params=params_best_optuna3)
metrics = per_era_metrics(validation, pred_col=model_name)
results_df = log_to_results(results_df, model_name=model_name, elapsed_time=elapsed_time, metrics=metrics, params=params_best_optuna3)
save_experiment(model=model, model_name=model_name, validation_data=validation, results_df=results_df)
results_df

In [ ]:
# Era-Wise CV for Model with Best Params Optuna
model_name = "best_optuna_cv"

cv_results, cv_mean, importance_df = era_wise_cv(
    train_df=train,
    feature_set=feature_set,
    params=params_best_optuna,
    n_splits=5,
    )


display(cv_results)
display(get_diff_cv(results_df, model_name_diff="best_optuna", model_name_cv="best_optuna_cv", cv_mean=cv_mean))

In [ ]:
df1 = pd.DataFrame({
    "model_name": ["baseline", "baseline_cv"],
    "corr_sharpe": [1.472, 2.114],
    "corr_mean": [0.031, 0.036],
    "corr_std": [0.021, 0.017]
})

In [ ]:
cv_mean_final = pd.concat([df1, cv_mean], ignore_index=True)

In [ ]:
with open("results_df.pkl", "wb") as f:
    pickle.dump(importance_df, f)

Продолжаем участие в классическом турнире Numerai.
1. Датасеты для обучения: Train (1865208 строк), Validation (3798341 строк). 737 признаков. 
2. Обучил baseline model: LGBM Regressor с параметрами из учебного ноутбука {"n_estimators": 20000, "learning_rate": 0.001, "max_depth": 6, "num_leaves": 64, "colsample_bytree": 0.1} 
3. Получил метрики baseline: corr_sharpe=1.472, corr_mean=0.031, corr_std=0.021 
4. Запустил для этой же модели с теми же параметрами era_wise_cv (5 фолдов) и получил сдедующий результат:
fold train_eras val_eras corr_sharpe corr_mean corr_std 
0 1 210-270 275-334 1.652 0.031 0.019 
1 2 210-330 335-394 1.926 0.029 0.015 
2 3 210-390 395-454 2.607 0.043 0.016 
3 4 210-450 455-514 1.859 0.035 0.019 
4 5 210-510 515-574 2.527 0.042 0.017
5. СРАВНЕНИЕ С БАЗОВОЙ МОДЕЛЬЮ:
model_name corr_sharpe corr_mean corr_std 
0 baseline 1.472 0.031 0.021 
1 baseline_cv 2.114 0.036 0.017
6. Запустил поиск best_iter на последнем фолде с ранней остановкой=100:
esr = 100 
X_train_es, y_train_es, X_val_es, y_val_es = get_last_fold(train, feature_set, fold=-1) 
model = lgb.LGBMRegressor(**params, random_state=SEED) 
model.fit( X_train_es, y_train_es, eval_set=[(X_val_es, y_val_es)], eval_names=["val_es"], callbacks=[early_stopping(stopping_rounds=esr), log_evaluation(period=0)] ) 
best_iter_last_fold = model.best_iteration_
И получил best_iter=8391 
7. Обучил модель "tuned_0" с params_tuned_0 = {
    "n_estimators": int(best_iter_last_fold * 1.2), 
    "learning_rate": 0.001,
    "max_depth": 6,
    "num_leaves": 64,
    "colsample_bytree": 0.1,
    "reg_alpha": 1.0,
    "reg_lambda": 1.0,
    "min_gain_to_split": 0.01,
    "subsample": 0.9,
    "verbosity": -1,
    "device_type": "gpu",}
8. Получил метрики модели "tuned_0" с params_tuned: corr_sharpe=1.351 , corr_mean=0.030, corr_std=0.022
9. Обучил модель "tuned" с params_tuned = {
    'n_estimators': 10000, # примерно best_iter   
    'learning_rate': 0.01,          
    'max_depth': 5,                  
    'num_leaves': 31,               
    'colsample_bytree': 0.05,        
    'subsample': 0.8,               
    'reg_alpha': 10.0,               
    'reg_lambda': 10.0,              
    'min_child_samples': 200,        
    'min_child_weight': 1e-2,        
    'verbosity': -1,
    'device_type': 'gpu'}
10. Получил результаты модели "tuned" с params_tuned: corr_sharpe=1.530, corr_mean=0.030 , corr_std=0.020

Как правильно интерпретировать полученные результаты? Какие выводы можно из этого сделать объясни подробно и почему так.
Основываясь на размерах датасетов, полученных метрик и best_iter, а также учитывая специфику турнира Numerai - напиши словарь параметров для Optuna, чтобы улучшить модель, но и адекватно было по вычислительным мощностям и времени.